# Comprehensive KPI/KPI Model Test
This notebook loads the full `kpi_okr_data_fixed.xlsx` dataset, performs exploratory analysis, constructs risk features, trains a KNN risk model, and evaluates results.

In [9]:
import sys
from pathlib import Path
project_path = Path('c:/Users/phuoc/PycharmProjects/PythonProject/kpiokrbe/OKR-KPI-system-AI')
if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))
print('Project path added:', project_path)
print('Working directory:', Path.cwd())

Project path added: c:\Users\phuoc\PycharmProjects\PythonProject\kpiokrbe\OKR-KPI-system-AI
Working directory: c:\Users\phuoc\PycharmProjects\PythonProject\kpiokrbe\OKR-KPI-system-AI


In [ ]:
# Install required notebook dependencies if needed
# Uncomment and run this cell only if packages are missing
# import subprocess, sys
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'openpyxl', 'scikit-learn', 'joblib'])

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from app.ml.knn.trainer import KNNModelTrainer
from app.ml.knn.predictor import KNNRiskPredictor
from datetime import datetime
print('Imports successful')

Imports successful


In [11]:
data_path = Path('kpi_okr_data_fixed.xlsx')
assert data_path.exists(), f'Dataset not found: {data_path}'
xl = pd.ExcelFile(data_path, engine='openpyxl')
print('Sheets available:', xl.sheet_names)
df_users = xl.parse('Users')
df_assignments = xl.parse('KPI_Assignments')
df_records = xl.parse('KPI_Records')
df_feedback = xl.parse('Feedbacks')
df_checkins = xl.parse('Check_Ins')
print('Users:', df_users.shape)
print('KPI Assignments:', df_assignments.shape)
print('KPI Records:', df_records.shape)
print('Feedbacks:', df_feedback.shape)
print('Check Ins:', df_checkins.shape)

Sheets available: ['Companies', 'Units', 'Users', 'Cycles', 'KPI_Dictionaries', 'KPI_Assignments', 'KPI_Records', 'Objectives', 'Key_Results', 'Check_Ins', 'Feedbacks', 'Notifications', 'Notification_Recipients', 'AI_Usage_Logs']
Users: (170, 12)
KPI Assignments: (797, 16)
KPI Records: (18694, 11)
Feedbacks: (479, 11)
Check Ins: (3562, 9)


## 1. Exploratory Data Analysis
Inspect the main sheets and understand the columns used for feature construction.

In [12]:
print('Users columns:', df_users.columns.tolist())
print('KPI Assignments columns:', df_assignments.columns.tolist())
print('KPI Records columns:', df_records.columns.tolist())
print('Feedbacks columns:', df_feedback.columns.tolist())
print('Check Ins columns:', df_checkins.columns.tolist())

Users columns: ['id', 'company_id', 'full_name', 'email', 'password', 'avatar_url', 'job_title', 'role', 'unit_id', 'is_active', 'deleted_at', 'created_at']
KPI Assignments columns: ['id', 'company_id', 'parent_assignment_id', 'kpi_dictionary_id', 'cycle_id', 'owner_id', 'unit_id', 'visibility', 'access_path', 'start_value', 'target_value', 'current_value', 'progress_percentage', 'due_date', 'deleted_at', 'created_at']
KPI Records columns: ['id', 'company_id', 'kpi_assignment_id', 'period_start', 'period_end', 'actual_value', 'progress_percentage', 'evidence_url', 'status', 'trend', 'created_at']
Feedbacks columns: ['id', 'company_id', 'objective_id', 'kr_tag_id', 'user_id', 'parent_id', 'content', 'sentiment', 'status', 'created_at', 'updated_at']
Check Ins columns: ['id', 'company_id', 'key_result_id', 'user_id', 'achieved_value', 'progress_snapshot', 'evidence_url', 'comment', 'created_at']


## 2. Feature Engineering
Build user-level risk features from KPI progress, feedback sentiment, and check-in activity.

In [13]:
# Map sentiment labels to numeric scores
sentiment_map = {'POSITIVE': 1.0, 'NEUTRAL': 0.0, 'NEGATIVE': -1.0, 'MIXED': 0.0}
df_feedback['sentiment_score'] = df_feedback['sentiment'].map(sentiment_map).fillna(0.0)

# KPI assignment-level completion and progress information by owner
user_kpi = df_assignments.groupby('owner_id').agg(
    kpi_completion_rate=('progress_percentage', 'mean'),
    kpi_targets=('target_value', 'count'),
    kpi_progress_std=('progress_percentage', 'std')
).reset_index()

# User feedback aggregates
user_feedback = df_feedback.groupby('user_id').agg(
    total_feedback=('id', 'count'),
    average_sentiment_score=('sentiment_score', 'mean'),
    positive_feedback_pct=('sentiment_score', lambda x: (x > 0).mean()),
    negative_feedback_pct=('sentiment_score', lambda x: (x < 0).mean()),
).reset_index()

# Check-in delay proxy using due dates from Key Results
df_key_results = xl.parse('Key_Results')
checkin_merged = df_checkins.merge(df_key_results[['id', 'due_date']], left_on='key_result_id', right_on='id', how='left', suffixes=('', '_kr'))
checkin_merged['delay_days'] = (checkin_merged['created_at'] - checkin_merged['due_date']).dt.days
checkin_merged['delay_days'] = checkin_merged['delay_days'].fillna(0.0)
user_checkins = checkin_merged.groupby('user_id').agg(
    avg_delay_days=('delay_days', 'mean'),
    total_checkins=('id', 'count'),
    avg_progress_snapshot=('progress_snapshot', 'mean')
).reset_index()

# Objective participation proxy
user_objectives = df_assignments.groupby('owner_id').agg(
    objective_participation_ratio=('owner_id', 'count')
).reset_index()

# Consolidate features into a single user-level dataset
dataset = user_kpi.merge(user_feedback, left_on='owner_id', right_on='user_id', how='left')
dataset = dataset.merge(user_checkins, left_on='owner_id', right_on='user_id', how='left', suffixes=('_kpi', '_checkin'))
dataset = dataset.merge(user_objectives, on='owner_id', how='left')

# Fill missing values with reasonable defaults
dataset['average_sentiment_score'] = dataset['average_sentiment_score'].fillna(0.0)
dataset['total_feedback'] = dataset['total_feedback'].fillna(0).astype(int)
dataset['positive_feedback_pct'] = dataset['positive_feedback_pct'].fillna(0.0)
dataset['negative_feedback_pct'] = dataset['negative_feedback_pct'].fillna(0.0)
dataset['avg_delay_days'] = dataset['avg_delay_days'].fillna(0.0)
dataset['total_checkins'] = dataset['total_checkins'].fillna(0).astype(int)
dataset['avg_progress_snapshot'] = dataset['avg_progress_snapshot'].fillna(0.0)
dataset['objective_participation_ratio'] = dataset['objective_participation_ratio'].fillna(0).astype(int)

dataset['objective_participation_ratio'] = dataset['objective_participation_ratio'] / dataset['objective_participation_ratio'].max()

dataset.head(5)

,owner_id,kpi_completion_rate,kpi_targets,kpi_progress_std,user_id_kpi,total_feedback,average_sentiment_score,positive_feedback_pct,negative_feedback_pct,user_id_checkin,avg_delay_days,total_checkins,avg_progress_snapshot,objective_participation_ratio
0,1,79.718667,30,24.801042,1.0,1,0.00,0.0,0.00,1,-46.461538,13,39.213077,0.243902
1,2,72.652857,7,33.143390,2.0,2,0.00,0.0,0.00,2,-46.600000,20,37.789000,0.056911
2,3,85.186000,5,12.244081,3.0,4,-0.25,0.0,0.25,3,-29.277778,18,53.984444,0.040650
3,4,76.310000,3,2.234211,4.0,2,0.50,0.5,0.00,4,-45.750000,16,38.336875,0.024390
4,5,76.815000,2,6.781154,5.0,1,0.00,0.0,0.00,5,-35.105263,19,48.498947,0.016260


In [14]:
# Create a derived risk label based on KPI completion, sentiment, and delay
def compute_risk_label(row):
    score = 0.0
    if row['kpi_completion_rate'] < 50:
        score += 0.35
    elif row['kpi_completion_rate'] < 70:
        score += 0.18
    if row['average_sentiment_score'] < 0:
        score += 0.25
    elif row['average_sentiment_score'] == 0:
        score += 0.10
    if row['avg_delay_days'] > 10:
        score += 0.25
    elif row['avg_delay_days'] > 3:
        score += 0.12
    if row['objective_participation_ratio'] < 0.5:
        score += 0.10
    if score >= 0.45:
        return 'high'
    if score >= 0.20:
        return 'medium'
    return 'low'

dataset['risk_label'] = dataset.apply(compute_risk_label, axis=1)
dataset['risk_score'] = dataset.apply(lambda row: 1.0 if row['risk_label'] == 'high' else (0.5 if row['risk_label'] == 'medium' else 0.1), axis=1)
dataset[['owner_id', 'kpi_completion_rate', 'average_sentiment_score', 'avg_delay_days', 'objective_participation_ratio', 'risk_label', 'risk_score']].head(10)

,owner_id,kpi_completion_rate,average_sentiment_score,avg_delay_days,objective_participation_ratio,risk_label,risk_score
0,1,79.718667,0.000000,-46.461538,0.243902,medium,0.5
1,2,72.652857,0.000000,-46.600000,0.056911,medium,0.5
2,3,85.186000,-0.250000,-29.277778,0.040650,medium,0.5
3,4,76.310000,0.500000,-45.750000,0.024390,low,0.1
4,5,76.815000,0.000000,-35.105263,0.016260,medium,0.5
5,6,78.765833,-0.333333,-43.700000,0.097561,medium,0.5
6,7,81.000000,0.000000,-37.375000,0.081301,medium,0.5
7,8,77.826000,0.400000,-43.882353,0.040650,low,0.1
8,9,92.260000,0.500000,-43.518519,0.008130,low,0.1
9,10,73.595000,0.333333,-50.818182,0.016260,low,0.1


## 3. Train / Test Split
Split the derived dataset and train a KNN classifier on the engineered features.

In [15]:
feature_cols = [
    'kpi_completion_rate',
    'average_sentiment_score',
    'avg_delay_days',
    'objective_participation_ratio'
]
X = dataset[feature_cols].fillna(0.0).to_numpy()
y = dataset['risk_label'].to_numpy()
label_counts = pd.Series(y).value_counts()
print('Label counts:', label_counts.to_dict())
if (label_counts < 2).any():
    print('Warning: Some classes have fewer than 2 samples. Splitting without stratification.')
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=None
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)
print('Label distribution:', pd.Series(y).value_counts(normalize=True).to_dict())

Label counts: {'medium': 52, 'low': 24, 'high': 1}
Train shape: (57, 4) Test shape: (20, 4)
Label distribution: {'medium': 0.6753246753246753, 'low': 0.3116883116883117, 'high': 0.012987012987012988}


In [16]:
# Train the KNN model and save artifacts
model = KNNModelTrainer.train_and_save(
    X_train,
    y_train,
    feature_cols,
    output_dir='models',
    n_neighbors=5,
    metric='euclidean'
)
print('Model trained and saved successfully')

PicklingError: Can't pickle <class 'app.ml.knn.trainer.KNNModelTrainer.train_and_save.<locals>.ModelWithScaler'>: it's not found as app.ml.knn.trainer.KNNModelTrainer.train_and_save.<locals>.ModelWithScaler

## 4. Evaluation
Evaluate model predictions and inspect classification performance.

In [ ]:
from sklearn.preprocessing import StandardScaler
X_train_scaled = model.scaler.transform(X_train)
X_test_scaled = model.scaler.transform(X_test)
y_pred = model.knn.predict(X_test_scaled)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Classification report:
', classification_report(y_test, y_pred))
print('Confusion matrix:
', confusion_matrix(y_test, y_pred))

## 5. Sample Predictions with Production Predictor
Test the saved KNN predictor on representative user feature rows.

In [ ]:
predictor = KNNRiskPredictor.get_instance(model_dir='models')
sample_rows = dataset.sample(5, random_state=1)
for idx, row in sample_rows.iterrows():
    # Use the same feature names used during training/saved in feature_columns.pkl
    features = {
        'kpi_completion_rate': float(row['kpi_completion_rate']),
        'avg_delay_days': float(row['avg_delay_days']),
        'average_sentiment_score': float(row['average_sentiment_score']),
        'objective_participation_ratio': float(row['objective_participation_ratio']),
    }
    label, score = predictor.predict(features)
    print(f'User {row["owner_id"]}: predicted={label}, score={score:.3f}, expected={row["risk_label"]}')

## 6. Summary
This notebook demonstrates how to load the provided KPI/OKR dataset, engineer features for risk prediction, train a KNN classifier, and validate the results with metrics and sample predictions.